In [1]:
!git clone https://github.com/code2expert/ai-datacenter-bootcamp-labs.git

Cloning into 'ai-datacenter-bootcamp-labs'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 93 (delta 19), reused 80 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 99.71 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [2]:
%run /content/ai-datacenter-bootcamp-labs/shared/python_venv.py

In [3]:
!/content/venv/bin/python --version

Python 3.10.12


In [4]:
import subprocess

subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install", "-q",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "autoawq==0.2.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

print("serving pins installed in Python 3.10 venv")

serving pins installed in Python 3.10 venv


In [5]:
import os, signal, subprocess

VENV_PYTHON = "/content/venv/bin/python"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args):
    cmd = [VENV_PYTHON, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args):
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 9008, logging to /content/server.log


In [6]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 0s: http://localhost:8000/v1/models -> 200


In [12]:
%cd /content/ai-datacenter-bootcamp-labs
!git pull

/content/ai-datacenter-bootcamp-labs
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.17 KiB | 2.08 MiB/s, done.
From https://github.com/code2expert/ai-datacenter-bootcamp-labs
   557127d..68b6e44  main       -> origin/main
Updating 557127d..68b6e44
Fast-forward
 w3d5-benchmark-harness/bench.py | 305 ++++++++++++++++++++++++++++++++++++++++
 1 file changed, 305 insertions(+)
 create mode 100644 w3d5-benchmark-harness/bench.py


In [13]:
%cd /content/ai-datacenter-bootcamp-labs/w3d5-benchmark-harness

/content/ai-datacenter-bootcamp-labs/w3d5-benchmark-harness


In [14]:
!/content/venv/bin/python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=92.53 ttft_p95=0.0701 errors=0
[level 2] tok/s=173.9 ttft_p95=0.0872 errors=0
[level 4] tok/s=290.37 ttft_p95=0.1016 errors=0
[level 8] tok/s=482.46 ttft_p95=0.159 errors=0
[level 16] tok/s=709.5 ttft_p95=0.2374 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     92.53      0.050      0.070     1.400    20     0
   2    173.90      0.058      0.087     1.476    20     0
   4    290.37      0.077      0.102     1.675    20     0
   8    482.46      0.135      0.159     1.878    20     0
  16    709.50      0.232      0.237     2.337    20     0

wrote bench_report.json (run appended)


In [15]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.0

under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None

print("knee:", knee)

c= 1  tok/s=   92.5  ttft_p95=0.070  lat_p95=1.400  errors=0
c= 2  tok/s=  173.9  ttft_p95=0.087  lat_p95=1.476  errors=0
c= 4  tok/s=  290.4  ttft_p95=0.102  lat_p95=1.675  errors=0
c= 8  tok/s=  482.5  ttft_p95=0.159  lat_p95=1.878  errors=0
c=16  tok/s=  709.5  ttft_p95=0.237  lat_p95=2.337  errors=0
knee: {'concurrency': 8, 'tokens_per_s': 482.46, 'ttft_p50_s': 0.1354, 'ttft_p95_s': 0.159, 'latency_p95_s': 1.878, 'errors': 0, 'ok': 20, 'wall_s': 4.303}


In [16]:
with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": TARGET_P95_S,
        "knee_concurrency": knee["concurrency"] if knee else None
    }, f, indent=2)

In [17]:
!cat knee.json

{
  "target_p95_s": 2.0,
  "knee_concurrency": 8
}

In [18]:
%%writefile capacity-note.md
# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (our SLO today): `2.0 seconds`
- Knee concurrency (highest concurrency whose p95 is still under target): `8`
- Tokens per second at the knee: `482.46 tokens/s`
- Max sustainable request rate at the target p95: `4.65 req/s`

## The limiting family

- Memory-bound: this workload is decode-heavy, where LLM serving is typically limited by memory movement rather than compute.

## Why the knee, not the peak

- The knee is the useful capacity because it keeps p95 within our 2.0-second SLO, while higher concurrency gives more throughput but causes latency to exceed the target.

Overwriting capacity-note.md


In [19]:
import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [20]:
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>